In [12]:
import altair as alt
import numpy as np
import pandas as pd

Loading the results dataset

In [13]:
results_data = pd.read_csv("results.csv")
results_data

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
49066,2026-01-18,Bolivia,Panama,1,1,Friendly,Tarija,Bolivia,False
49067,2026-01-18,Grenada,Jamaica,0,1,Friendly,St. George's,Grenada,False
49068,2026-01-22,Panama,Mexico,0,1,Friendly,Panama City,Panama,False
49069,2026-01-25,Bolivia,Mexico,0,1,Friendly,Santa Cruz,Bolivia,False


Sorting the results data to only consider obervations(matches) from 2015 onwards

In [14]:
# Only considering matches starting after 2015
results_data['date'] = pd.to_datetime(results_data['date'])
results_filtered = results_data[results_data['date'] >= '2015-01-01']
results_filtered

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
38380,2015-01-04,Bahrain,Jordan,1,0,Friendly,Ballarat,Australia,True
38381,2015-01-04,Iran,Iraq,1,0,Friendly,Wollongong,Australia,True
38382,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,Parramatta,Australia,True
38383,2015-01-04,South Africa,Zambia,1,0,Friendly,Johannesburg,South Africa,False
38384,2015-01-05,China PR,Oman,4,1,Friendly,Penrith,Australia,True
...,...,...,...,...,...,...,...,...,...
49066,2026-01-18,Bolivia,Panama,1,1,Friendly,Tarija,Bolivia,False
49067,2026-01-18,Grenada,Jamaica,0,1,Friendly,St. George's,Grenada,False
49068,2026-01-22,Panama,Mexico,0,1,Friendly,Panama City,Panama,False
49069,2026-01-25,Bolivia,Mexico,0,1,Friendly,Santa Cruz,Bolivia,False


Loading the rankings dataset and Converting the rank_date variable into a 'date' for python

In [15]:
rankings_data = pd.read_csv("fifa_ranking-2024-06-20.csv")
rankings_data["year"] = pd.to_datetime(rankings_data["rank_date"]).dt.year
rankings_data


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date,year
0,140.0,Brunei Darussalam,BRU,2.00,0.00,140,AFC,1992-12-31,1992
1,33.0,Portugal,POR,38.00,0.00,33,UEFA,1992-12-31,1992
2,32.0,Zambia,ZAM,38.00,0.00,32,CAF,1992-12-31,1992
3,31.0,Greece,GRE,38.00,0.00,31,UEFA,1992-12-31,1992
4,30.0,Algeria,ALG,39.00,0.00,30,CAF,1992-12-31,1992
...,...,...,...,...,...,...,...,...,...
67467,137.0,Kuwait,KUW,1098.42,1085.46,-2,AFC,2024-06-20,2024
67468,136.0,Lithuania,LTU,1100.66,1095.23,-1,UEFA,2024-06-20,2024
67469,135.0,Malaysia,MAS,1107.58,1094.54,-3,AFC,2024-06-20,2024
67470,133.0,Solomon Islands,SOL,1111.02,1111.02,1,OFC,2024-06-20,2024


Filter the rankings data to only consider rankings from 2015 onwards and for each country to only have 1 rankings per year

In [16]:
# Only considering rankings starting after 2015 and only have 1 observation per year per team
rankings_filtered = (
    rankings_data[rankings_data['year'] >= 2015]
    .groupby(['country_full', 'year'])
    .last()
    .reset_index()
)
rankings_filtered

,country_full,year,rank,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,Afghanistan,2015,150.0,AFG,194.00,168.00,-6,AFC,2015-12-03
1,Afghanistan,2016,146.0,AFG,189.00,189.00,-1,AFC,2016-12-22
2,Afghanistan,2017,148.0,AFG,181.00,181.00,1,AFC,2017-12-21
3,Afghanistan,2018,147.0,AFG,1068.00,1068.00,0,AFC,2018-12-20
4,Afghanistan,2019,149.0,AFG,1052.00,1052.00,0,AFC,2019-12-19
...,...,...,...,...,...,...,...,...,...
2101,Zimbabwe,2020,108.0,ZIM,1181.00,1181.00,0,CAF,2020-12-10
2102,Zimbabwe,2021,121.0,ZIM,1138.44,1138.44,0,CAF,2021-12-23
2103,Zimbabwe,2022,125.0,ZIM,1138.56,1138.56,0,CAF,2022-12-22
2104,Zimbabwe,2023,124.0,ZIM,1144.56,1144.56,0,CAF,2023-12-21


Teams that have already qualified for the 2026 world cup

In [17]:
world_cup_2026_teams = [
    # Host nations (auto-qualified)
    "United States", "Canada", "Mexico",
    # UEFA
    "Germany", "Portugal", "Spain", "France", "England", "Netherlands",
    "Belgium", "Austria", "Turkey", "Poland", "Serbia",
    # CONMEBOL
    "Argentina", "Colombia", "Uruguay", "Brazil", "Ecuador", "Paraguay",
    # CONCACAF
    "Panama", "Costa Rica", "Honduras", "Jamaica",
    # CAF
    "Morocco", "Egypt", "Senegal", "South Africa", "DR Congo",
    "Ivory Coast", "Nigeria", "Algeria", "Tunisia",
    # AFC
    "Iran", "South Korea", "Japan", "Saudi Arabia", "Australia",
    "Uzbekistan", "Jordan", "Iraq",
    # OFC
    "New Zealand",
]

Filter to only include matches playd between two teams that have qualified for the 2026 world cup

In [18]:
# Only includes teams currently qualified for the wolrd cup
results_wc2026 = results_filtered[
    results_filtered['home_team'].isin(world_cup_2026_teams) &
    results_filtered['away_team'].isin(world_cup_2026_teams)
]
results_wc2026

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
38381,2015-01-04,Iran,Iraq,1,0,Friendly,Wollongong,Australia,True
38382,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,Parramatta,Australia,True
38395,2015-01-11,Nigeria,Ivory Coast,0,1,Friendly,Abu Dhabi,United Arab Emirates,True
38396,2015-01-11,Tunisia,Algeria,1,1,Friendly,Radès,Tunisia,False
38399,2015-01-12,Jordan,Iraq,0,1,AFC Asian Cup,Brisbane,Australia,True
...,...,...,...,...,...,...,...,...,...
49062,2026-01-14,Senegal,Egypt,1,0,African Cup of Nations,Tangier,Morocco,True
49063,2026-01-14,Morocco,Nigeria,0,0,African Cup of Nations,Rabat,Morocco,False
49064,2026-01-17,Egypt,Nigeria,0,0,African Cup of Nations,Casablanca,Morocco,True
49065,2026-01-18,Morocco,Senegal,0,1,African Cup of Nations,Rabat,Morocco,False


Filter to only include rankings for teams that are qualified for the 2026 world cup

In [19]:
# Only includes teams currently qualified for the wolrd cup
rankings_wc2026 = rankings_filtered[
    rankings_filtered['country_full'].isin(world_cup_2026_teams)
].sort_values(by=['rank_date', 'rank'], ascending=[False, True])
rankings_wc2026

,country_full,year,rank,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
89,Argentina,2024,1.0,ARG,1860.14,1858.00,0,CONMEBOL,2024-06-20
717,France,2024,2.0,FRA,1837.47,1840.59,0,UEFA,2024-06-20
199,Belgium,2024,3.0,BEL,1797.98,1795.23,0,UEFA,2024-06-20
279,Brazil,2024,4.0,BRA,1791.85,1788.65,-1,CONMEBOL,2024-06-20
627,England,2024,5.0,ENG,1787.88,1794.90,1,UEFA,2024-06-20
...,...,...,...,...,...,...,...,...,...
967,Jordan,2015,87.0,JOR,399.00,411.00,5,AFC,2015-12-03
360,Canada,2015,88.0,CAN,388.00,335.00,-14,CONCACAF,2015-12-03
917,Iraq,2015,89.0,IRQ,381.00,392.00,2,AFC,2015-12-03
847,Honduras,2015,101.0,HON,338.00,359.00,6,CONCACAF,2015-12-03


Checking more missing values

In [20]:
# check for missing values
print(results_filtered.isnull().sum())
print(rankings_filtered.isnull().sum())


date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64
country_full       0
year               0
rank               1
country_abrv       0
total_points       0
previous_points    0
rank_change        0
confederation      0
rank_date          0
dtype: int64


In [21]:
# Test